# DINOv3 Searchlight RSA

**Pipeline:**
1. Authenticate with HuggingFace (gated model)
2. Load IAPS images (60 stimuli, converted to grayscale)
3. Extract image embeddings (CLS token) from DINOv3 ViT-B/16
4. Build model RDM
5. Run whole-brain searchlight RSA (20 subjects)
6. Group-level t-maps at FDR 0.05, 0.01, 0.001

DINOv3 is purely self-supervised (no text, no emotion labels).

**Prerequisites:**
- `pip install transformers`
- Accept the license at https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m
- Log in via the cell below

In [1]:
import os
import numpy as np
import pandas as pd
import pickle
import torch
from PIL import Image
from glob import glob
from sklearn.metrics import pairwise_distances
from tqdm import tqdm
import matplotlib.pyplot as plt
import nibabel as nib
from nilearn.image import new_img_like
import nilearn.image as nlimg
from nilearn import plotting
from scipy.stats import ttest_1samp, spearmanr
from statsmodels.stats.multitest import fdrcorrection
from rsatoolbox.util.searchlight import get_volume_searchlight, get_searchlight_RDMs
from rsatoolbox.util.searchlight import evaluate_models_searchlight
from rsatoolbox.model import ModelFixed
from rsatoolbox.inference import eval_fixed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

os.makedirs('./outputs/tBrainmap', exist_ok=True)
os.makedirs('./outputs/embeddings', exist_ok=True)

Device: cpu


In [2]:
# =====================================================================
# HuggingFace authentication (required for gated DINOv3)
# =====================================================================
# Fix SSL_CERT_FILE issue if present
if 'SSL_CERT_FILE' in os.environ:
    del os.environ['SSL_CERT_FILE']

from huggingface_hub import login
login()  # paste your token when prompted

In [3]:
# =====================================================================
# CONFIG
# =====================================================================

# Directory containing the 60 IAPS images
IAPS_IMAGE_DIR = r"N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\data\raw_iaps"
# ^^^ UPDATE if your images are elsewhere

# IAPS image order file
IAPS_ORDER_CSV = r"..\IAPS_Searchlight\IAPS_60_pnu.csv"

# fMRI searchlight data
FMRI_DATA_DIR = r"."
TMP_IMG_PATH  = os.path.join(FMRI_DATA_DIR, 'tmp.img')
MASK_PATH     = os.path.join(FMRI_DATA_DIR, 'mask.npy')
ALLSUB_PATH   = os.path.join(FMRI_DATA_DIR, 'allsub_avg.npy')

# Searchlight parameters
SL_RADIUS    = 5
SL_THRESHOLD = 0.5
N_JOBS       = 6

# Statistics
FDR_ALPHAS        = [0.05, 0.01, 0.001]
CLUSTER_THRESHOLD = 30

N_STIMULI = 60

In [4]:
def upper_tri(RDM):
    m = RDM.shape[0]
    r, c = np.triu_indices(m, 1)
    return RDM[r, c]


def rdm_from_features(features, metric='correlation'):
    rdm = pairwise_distances(features, metric=metric)
    np.fill_diagonal(rdm, 0)
    return rdm


def normalize_rdm(rdm):
    mn, mx = rdm.min(), rdm.max()
    if mx - mn == 0:
        return rdm
    return (rdm - mn) / (mx - mn)

## 1. Load IAPS images in stimulus order

In [5]:
iaps_order = pd.read_csv(IAPS_ORDER_CSV)
iaps_ids = [str(x) for x in iaps_order['iaps_id']]
print(f'IAPS order: {len(iaps_ids)} images')

all_files = glob(os.path.join(IAPS_IMAGE_DIR, '*'))
file_lookup = {}
for f in all_files:
    base = os.path.splitext(os.path.basename(f))[0]
    file_lookup[base] = f

images_pil = []
for iid in iaps_ids:
    if iid in file_lookup:
        img = Image.open(file_lookup[iid]).convert('L').convert('RGB')  # grayscale -> 3ch
        images_pil.append(img)
    else:
        raise FileNotFoundError(f'Image not found for IAPS ID: {iid}')

print(f'Loaded {len(images_pil)} images')
print(f'Sample size: {images_pil[0].size}')

FileNotFoundError: [Errno 2] No such file or directory: '..\\IAPS_Searchlight\\IAPS_60_pnu.csv'

## 2. Extract DINOv3 embeddings

In [ ]:
from transformers import AutoImageProcessor, AutoModel

DINOV3_MODEL_ID = 'facebook/dinov3-vitb16-pretrain-lvd1689m'

print('Loading DINOv3 ViT-B/16...')
dinov3_processor = AutoImageProcessor.from_pretrained(DINOV3_MODEL_ID)
dinov3_model = AutoModel.from_pretrained(DINOV3_MODEL_ID).eval().to(device)
print('Model loaded.')

dinov3_embeddings = []
with torch.no_grad():
    for img in tqdm(images_pil, desc='DINOv3'):
        inputs = dinov3_processor(images=img, return_tensors='pt').to(device)
        outputs = dinov3_model(**inputs)
        # DINOv3 pooler_output is properly trained
        cls_token = outputs.pooler_output  # (1, 768)
        dinov3_embeddings.append(cls_token.cpu().numpy().flatten())

dinov3_embeddings = np.array(dinov3_embeddings)
print(f'DINOv3 embeddings: {dinov3_embeddings.shape}')  # (60, 768)

# Save embeddings
np.save('./outputs/embeddings/DINOv3_vitb16_embeddings.npy', dinov3_embeddings)
print('Saved: DINOv3_vitb16_embeddings.npy')

del dinov3_model, dinov3_processor
torch.cuda.empty_cache() if device == 'cuda' else None

## 3. Build RDM

In [ ]:
dinov3_rdm = rdm_from_features(dinov3_embeddings, metric='correlation')
print(f'DINOv3 RDM: {dinov3_rdm.shape}, range [{dinov3_rdm.min():.4f}, {dinov3_rdm.max():.4f}]')

# Save RDM
pd.DataFrame(dinov3_rdm, index=iaps_ids, columns=iaps_ids).to_csv(
    './outputs/rdm_DINOv3_vitb16.csv')
print('Saved: rdm_DINOv3_vitb16.csv')

# Compare with DINOv1 and DINOv2 if available
for other in ['DINOv1_vitb16', 'DINOv2_vitb14']:
    emb_path = f'./outputs/embeddings/{other}_embeddings.npy'
    if os.path.exists(emb_path):
        other_emb = np.load(emb_path)
        other_rdm = rdm_from_features(other_emb, metric='correlation')
        r, p = spearmanr(upper_tri(dinov3_rdm), upper_tri(other_rdm))
        print(f'DINOv3 vs {other}: r = {r:.3f}  (p = {p:.2e})')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5.5, 4.5))
im = ax.imshow(normalize_rdm(dinov3_rdm), cmap='viridis', aspect='equal')
ax.set_title('DINOv3 ViT-B/16 RDM', fontsize=11)
ax.set_xlabel('Stimulus')
ax.set_ylabel('Stimulus')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('./outputs/DINOv3_RDM.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Load fMRI data and build searchlight

In [ ]:
tmp_img = nib.load(TMP_IMG_PATH)
mask = np.load(MASK_PATH)
allsub_avg = np.load(ALLSUB_PATH, allow_pickle=True)
centers, neighbors = get_volume_searchlight(mask, radius=SL_RADIUS, threshold=SL_THRESHOLD)

print(f'Mask shape: {mask.shape}')
print(f'Subject data shape: {allsub_avg.shape}')
print(f'Searchlight centers: {len(centers)}')

## 5. Searchlight RSA functions

In [ ]:
def run_searchlight_for_model(model_name, model_rdm, allsub_avg, centers, neighbors, mask):
    model_obj = ModelFixed(model_name, upper_tri(model_rdm))

    eval_score_list = []
    RDM_brain_list = []
    x, y, z = mask.shape

    for i in tqdm(range(allsub_avg.shape[0]), desc=f'{model_name}'):
        subj_data = np.nan_to_num(allsub_avg[i])
        image_value = np.arange(subj_data.shape[0])

        SL_RDM = get_searchlight_RDMs(subj_data, centers, neighbors, image_value, method='correlation')

        eval_results = evaluate_models_searchlight(SL_RDM, model_obj, eval_fixed, method='spearman', n_jobs=N_JOBS)
        eval_score = [float(e.evaluations) for e in eval_results]

        RDM_brain = np.zeros(x * y * z)
        RDM_brain[list(SL_RDM.rdm_descriptors['voxel_index'])] = eval_score
        RDM_brain = RDM_brain.reshape(x, y, z)

        eval_score_list.append(eval_score)
        RDM_brain_list.append(RDM_brain)

    voxel_indices = np.array(SL_RDM.rdm_descriptors['voxel_index'])
    return np.array(eval_score_list), RDM_brain_list, voxel_indices


def compute_tmap_from_scores(eval_scores, voxel_indices, mask_shape, tmp_img,
                             fdr_alpha=0.001, cluster_threshold=30,
                             model_name='', t_value_threshold=None):
    epsilon = 1e-10
    scores_clipped = np.clip(eval_scores, -1 + epsilon, 1 - epsilon)
    fisher_z = np.arctanh(scores_clipped)

    t_stats, p_values = ttest_1samp(fisher_z, 0, axis=0, nan_policy='omit')

    rejected, p_fdr = fdrcorrection(p_values, alpha=fdr_alpha)
    sig_mask = rejected

    if t_value_threshold is not None:
        sig_mask = sig_mask & (t_stats >= t_value_threshold)
        print(f"{model_name}: {np.sum(sig_mask)} significant voxels "
              f"(FDR q < {fdr_alpha} AND t >= {t_value_threshold})")
    else:
        print(f"{model_name}: {np.sum(sig_mask)} significant voxels "
              f"(FDR q < {fdr_alpha})")

    sig_voxels = np.where(sig_mask)[0]

    x, y, z = mask_shape
    t_map = np.full(x * y * z, np.nan)
    t_map[voxel_indices[sig_voxels]] = t_stats[sig_voxels]
    t_map_3d = t_map.reshape(x, y, z)

    t_map_3d[t_map_3d < 0] = np.nan

    t_img = new_img_like(tmp_img, t_map_3d)

    t_img_cluster = nlimg.threshold_img(
        t_img,
        threshold=0,
        cluster_threshold=cluster_threshold
    )

    return t_img_cluster, t_stats, p_fdr

## 6. Run searchlight RSA

In [ ]:
model_name = 'DINOv3_vitb16'

print(f"\n{'='*60}")
print(f"  Running searchlight for: {model_name}")
print(f"{'='*60}")

eval_scores, rdm_brains, voxel_ids = run_searchlight_for_model(
    model_name, dinov3_rdm, allsub_avg, centers, neighbors, mask
)

pd.DataFrame(eval_scores).to_csv(f'./outputs/20sub_SL_eval_{model_name}.csv')
np.save(f'./outputs/20sub_SL_eval_{model_name}.npy', eval_scores)
print(f'Saved. Shape: {eval_scores.shape}')

## 7. Group statistics and t-maps (FDR 0.05, 0.01, 0.001)

In [ ]:
tmaps = {}

print(f"\n{'='*60}")
print(f"  {model_name}")
print(f"{'='*60}")

for fdr_alpha in FDR_ALPHAS:
    fdr_label = str(fdr_alpha).replace('0.', '').replace('.', '')

    tmap, t_stats, p_fdr = compute_tmap_from_scores(
        eval_scores, voxel_ids, mask.shape, tmp_img,
        fdr_alpha=fdr_alpha,
        cluster_threshold=CLUSTER_THRESHOLD,
        model_name=model_name,
        t_value_threshold=None
    )

    out_path = f'./outputs/tBrainmap/tmap_{model_name}_fdr{fdr_label}.nii.gz'
    nib.save(tmap, out_path)
    print(f'  Saved: {out_path}')

    tmaps[fdr_alpha] = tmap

## 8. Visualize t-maps

In [ ]:
# Interactive glass brain (FDR 0.001)
view = plotting.view_img(tmaps[0.001], draw_cross=False, black_bg=False, threshold=0,
                         title=f'{model_name} (FDR < 0.001)')
display(view)

In [ ]:
# Static slice views across FDR thresholds
n_fdrs = len(FDR_ALPHAS)
fig, axes = plt.subplots(1, n_fdrs, figsize=(6 * n_fdrs, 4.5))

for col, fdr_alpha in enumerate(FDR_ALPHAS):
    plotting.plot_stat_map(
        tmaps[fdr_alpha], display_mode='z', cut_coords=6,
        title=f'{model_name}\nFDR < {fdr_alpha}',
        axes=axes[col],
        threshold=0, colorbar=True
    )

plt.tight_layout()
plt.savefig('./outputs/tmap_DINOv3_all_fdr.png', dpi=150)
plt.show()

## 9. Reload saved results (if restarting)

In [ ]:
# # Reload
# dinov3_embeddings = np.load('./outputs/embeddings/DINOv3_vitb16_embeddings.npy')
# dinov3_rdm = rdm_from_features(dinov3_embeddings, metric='correlation')
# eval_scores = np.load('./outputs/20sub_SL_eval_DINOv3_vitb16.npy')
# print(f'Reloaded: {eval_scores.shape}')